# SSL Embeddings (IEMOCAP)

This notebook extracts pooled self-supervised embeddings (HuBERT/wav2vec2).
Each utterance becomes one training row for downstream SER models.

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd


In [14]:
# Configuration
REPO_ROOT = Path.cwd().parents[1]  # repo root (notebook is under feature_extraction/)
CSV_PATH = REPO_ROOT / "datasets" / "IEMOCAP" / "iemocap_full_dataset.csv"
AUDIO_ROOT = REPO_ROOT / "datasets" / "IEMOCAP"
OUT_DIR = REPO_ROOT / "extracted_features" / "ssl_embeddings"
OUT_FILE = "ssl_embeddings_features.csv"

# SSL params
MODEL_NAME = "facebook/hubert-base-ls960"
TARGET_SR = 16_000
DEVICE = None  # "cuda" or "cpu"; None selects automatically
POOL = ("mean", "std")

OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / OUT_FILE
OUT_PATH


WindowsPath('c:/Users/marsh/Documents/GitHub/Speech-Emotion-Recognition/extracted_features/ssl_embeddings/ssl_embeddings_features.csv')

In [ ]:
_MODEL_CACHE: dict[str, tuple[object, object]] = {}


def _require_torch_stack():
    try:
        import torch
        from transformers import AutoModel, AutoFeatureExtractor
    except ImportError as exc:
        raise ImportError(
            "SSL embeddings require torch and transformers. "
            "Install with: uv add torch transformers"
        ) from exc
    return torch, (AutoModel, AutoFeatureExtractor)

def _get_model(model_name: str, device: str | None):
    torch, (AutoModel, AutoFeatureExtractor) = _require_torch_stack()

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    cached = _MODEL_CACHE.get(model_name)
    if cached is None:
        fe = AutoFeatureExtractor.from_pretrained(model_name)
        model = AutoModel.from_pretrained(model_name)
        model.eval()
        _MODEL_CACHE[model_name] = (fe, model)
    else:
        fe, model = cached

    model = model.to(device)
    return fe, model, device


def _resample(audio: np.ndarray, sr: int, target_sr: int) -> np.ndarray:
    # Resample audio to target_sr, preferring torchaudio if available
    if sr == target_sr:
        return audio.astype(np.float32, copy=False)

    try:
        import torch
        import torchaudio

        waveform = torch.from_numpy(audio.astype(np.float32, copy=False))
        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)
        resampler = torchaudio.transforms.Resample(
            orig_freq=sr,
            new_freq=target_sr,
        )
        resampled = resampler(waveform).squeeze(0).cpu().numpy()
        return resampled.astype(np.float32, copy=False)
    except Exception:
        import librosa

        return librosa.resample(
            audio.astype(np.float32, copy=False),
            orig_sr=sr,
            target_sr=target_sr,
        ).astype(np.float32, copy=False)


def extract_ssl_embeddings(
    audio: np.ndarray,
    sr: int,
    *,
    model_name: str,
    device: str | None,
    target_sr: int,
    pool: tuple[str, ...],
) -> dict[str, np.ndarray]:
    torch, _ = _require_torch_stack()
    fe, model, device = _get_model(model_name, device)

    audio = np.asarray(audio, dtype=np.float32)
    audio = _resample(audio, sr, target_sr)

    inputs = fe(audio, sampling_rate=target_sr, return_tensors="pt", padding=True)
    input_values = inputs["input_values"].to(device)

    with torch.no_grad():
        outputs = model(input_values)
        hidden = outputs.last_hidden_state[0]

    frames, dim = hidden.shape
    features: dict[str, np.ndarray] = {
        "ssl_frames": np.asarray(frames, dtype=np.int64),
        "ssl_dim": np.asarray(dim, dtype=np.int64),
    }

    if "mean" in pool:
        mean_vec = hidden.mean(dim=0).float().cpu().numpy().astype(np.float32, copy=False)
        features["ssl_mean"] = mean_vec
    if "std" in pool:
        std_vec = hidden.std(dim=0, unbiased=False).float().cpu().numpy().astype(
            np.float32,
            copy=False,
        )
        features["ssl_std"] = std_vec

    return features


def flatten_embeddings(prefix: str, vec: np.ndarray) -> dict[str, float]:
    flat: dict[str, float] = {}
    for idx, value in enumerate(vec):
        flat[f"{prefix}_{idx:04d}"] = float(value)
    return flat


def extract_ssl_features(audio: np.ndarray, sr: int) -> dict[str, float]:
    emb = extract_ssl_embeddings(
        audio,
        sr,
        model_name=MODEL_NAME,
        device=DEVICE,
        target_sr=TARGET_SR,
        pool=POOL,
    )

    features: dict[str, float] = {
        "ssl_frames": float(emb["ssl_frames"]),
        "ssl_dim": float(emb["ssl_dim"]),
    }
    if "ssl_mean" in emb:
        features.update(flatten_embeddings("ssl_mean", emb["ssl_mean"]))
    if "ssl_std" in emb:
        features.update(flatten_embeddings("ssl_std", emb["ssl_std"]))
    return features


In [16]:
df = pd.read_csv(CSV_PATH)  # metadata for paths + labels
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()

# Filter: valid emotion + agreement > 0
df = df[(df["emotion"] != "xxx") & (df["agreement"] > 0)].copy()
df.shape


(7532, 7)

In [17]:
import librosa

rows: list[dict[str, float | str | int]] = []
missing: list[str] = []

for _, row in df.iterrows():
    rel_path = row["path"]
    audio_path = AUDIO_ROOT / rel_path
    if not audio_path.exists():
        missing.append(str(audio_path))
        continue

    audio, sr = librosa.load(audio_path, sr=TARGET_SR, mono=True)
    duration_s = audio.shape[0] / sr
    features = extract_ssl_features(audio, sr)

    record: dict[str, float | str | int] = {
        "path": str(rel_path),
        "session": int(row["session"]),
        "method": row["method"],
        "gender": row["gender"],
        "emotion": row["emotion"],
        "n_annotators": int(row["n_annotators"]),
        "agreement": int(row["agreement"]),
        "duration_s": float(duration_s),
    }
    record.update(features)
    rows.append(record)

feature_df = pd.DataFrame(rows)
feature_df.to_csv(OUT_PATH, index=False)

print(f"Saved: {OUT_PATH}")
if missing:
    print(f"Missing audio files: {len(missing)}")
feature_df.shape


TypeError: expected str, bytes or os.PathLike object, not NoneType